# 02 - Auditoria dos dados brutos
Objetivo: medir quantos registros cada tipo de problema afeta, por arquivo e por coluna. Esses números são o "antes" que comprova a limpeza e formam o dicionário de inconsistências do README.


In [1]:
import pandas as pd

compras = pd.read_csv("../data/raw/compras.csv", dtype=str)
itens = pd.read_csv("../data/raw/itens.csv", dtype=str)
resultados = pd.read_csv("../data/raw/resultados.csv", dtype=str)


---
## 1. Documentos (CNPJ/CPF)


In [2]:
def classificar_documento(doc, tipo):
    if pd.isna(doc):
        return "vazio"
    if tipo == "PE":
        return "estrangeiro"
    tamanho = 14 if tipo == "PJ" else 11
    if doc != doc.strip():
        return "espaços extras"
    if not doc.isdigit():
        return "máscara"
    if len(doc) < tamanho:
        return "zeros perdidos"
    if len(doc) == tamanho:
        return "limpo"
    return "dígitos demais"


In [3]:
compras["cnpj_orgao"].apply(lambda doc: classificar_documento(doc, "PJ")).value_counts()


cnpj_orgao
limpo             1658
máscara            176
espaços extras     131
zeros perdidos      77
Name: count, dtype: int64

In [4]:
situacao_ni = [classificar_documento(doc, tipo)
               for doc, tipo in zip(resultados["ni_fornecedor"], resultados["tipo_pessoa"])]

pd.crosstab(pd.Series(situacao_ni, name="situação"), resultados["tipo_pessoa"], margins=True)


tipo_pessoa,PE,PF,PJ,All
situação,,,,
espaços extras,0,293,695,988
estrangeiro,21,0,0,21
limpo,0,3687,9216,12903
máscara,0,392,1239,1631
zeros perdidos,0,159,194,353
All,21,4531,11344,15896


**Resultado:**

| Coluna | Limpo | Máscara | Espaços extras | Zeros perdidos | Estrangeiro | Total |
|---|---|---|---|---|---|---|
| `cnpj_orgao` | 1.658 | 176 | 131 | 77 | – | 2.042 |
| `ni_fornecedor` (PJ) | 9.216 | 1.239 | 695 | 194 | – | 11.344 |
| `ni_fornecedor` (PF) | 3.687 | 392 | 293 | 159 | – | 4.531 |
| `ni_fornecedor` (PE) | – | – | – | – | 21 | 21 |

19% dos documentos de fornecedor e 19% dos CNPJs de órgão têm algum problema. Sem a limpeza, o mesmo fornecedor apareceria com documentos diferentes: no bruto há 11.418 documentos distintos, e após padronizar sobram 10.299.


---
## 2. Valores monetários e numéricos


In [5]:
def auditar_numero(coluna):
    pontos = coluna.str.count(r"\.")
    virgulas = coluna.str.count(",")
    return pd.Series({
        "vazio": coluna.isna().sum(),
        "já no formato numérico (ex.: 3.6)": coluna.str.fullmatch(r"\d+(\.\d+)?").sum(),
        "com R$": coluna.str.contains(r"R\$", na=False).sum(),
        "espaços extras ou invisíveis": coluna.str.contains(r"^\s|\s$|\xa0", na=False).sum(),
        "negativo entre parênteses": coluna.str.contains(r"\(", na=False).sum(),
        "vírgula como decimal (ex.: 3,54)": ((virgulas == 1) & (pontos == 0)).sum(),
        "separador de milhar": ((pontos > 1) | (virgulas > 1) | ((pontos > 0) & (virgulas > 0))).sum(),
    })


In [6]:
auditoria_numeros = pd.DataFrame({
    "itens.quantidade": auditar_numero(itens["quantidade"]),
    "itens.valor_unitario_estimado": auditar_numero(itens["valor_unitario_estimado"]),
    "itens.valor_total_estimado": auditar_numero(itens["valor_total_estimado"]),
    "resultados.quantidade_homologada": auditar_numero(resultados["quantidade_homologada"]),
    "resultados.valor_unitario_homologado": auditar_numero(resultados["valor_unitario_homologado"]),
    "resultados.valor_total_homologado": auditar_numero(resultados["valor_total_homologado"]),
}).T

auditoria_numeros


,vazio,já no formato numérico (ex.: 3.6),com R$,espaços extras ou invisíveis,negativo entre parênteses,"vírgula como decimal (ex.: 3,54)",separador de milhar
itens.quantidade,0,5596,1148,551,0,1262,219
itens.valor_unitario_estimado,0,5475,1188,561,0,953,649
itens.valor_total_estimado,0,5415,1171,588,0,359,1303
resultados.quantidade_homologada,0,12472,2561,1267,0,2245,1179
resultados.valor_unitario_homologado,0,12282,2650,1352,0,2077,1537
resultados.valor_total_homologado,0,12064,2647,1332,0,690,3142


**Resultado:** cada coluna da tabela conta uma característica de forma independente. Um mesmo valor pode ter várias (ex.: `" R$ 1.234,56"` tem espaço, `R$` e milhar), por isso a soma de uma linha passa do total de registros.

- Cerca de 1 em cada 4 valores **não** está no formato numérico do Python e precisa de conversão.
- `R$` aparece também nas **quantidades** (1.148 em `itens` e 2.561 em `resultados`): formatação de moeda aplicada em campo errado no sistema de origem.
- Há espaços invisíveis (`\xa0`), vírgula decimal e separadores de milhar no padrão brasileiro e no americano em todas as 6 colunas.
- **Não há** valores negativos entre parênteses nesta base, embora o enunciado cite o caso.
- Nenhum valor vazio nas 6 colunas.

Regra de conversão (regra 12): um símbolo sozinho é sempre o decimal (`746.815` = 746,815); símbolo repetido é milhar; com os dois, o último é o decimal.


---
## 3. Datas


In [7]:
formato_publicacao = compras["data_publicacao"].str.replace(r"\d", "9", regex=True).value_counts()
formato_resultado = resultados["data_resultado"].str.replace(r"\d", "9", regex=True).value_counts()

pd.DataFrame({"data_publicacao": formato_publicacao, "data_resultado": formato_resultado}).fillna(0).astype(int)


,data_publicacao,data_resultado
99-99-9999,138,912
99.99.9999,120,937
99/99/9999,117,929
99/99/9999 99:99,126,934
9999-99-99,0,10301
9999-99-99T99:99:99,1319,0
9999/99/99,116,960
99999999,106,923


In [8]:
from datetime import datetime

FORMATOS_DATA = ["%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%d.%m.%Y", "%Y/%m/%d", "%Y%m%d"]

def converter_data(texto):
    if pd.isna(texto):
        return None
    texto = texto.strip()[:10]
    for formato in FORMATOS_DATA:
        try:
            return datetime.strptime(texto, formato).date()
        except ValueError:
            continue
    return None

publicacao = compras["data_publicacao"].map(converter_data)
homologacao = resultados["data_resultado"].map(converter_data)

print("Publicações não convertidas:", publicacao.isna().sum())
print("Homologações não convertidas:", homologacao.isna().sum())



Publicações não convertidas: 0
Homologações não convertidas: 0


In [9]:
datas = pd.DataFrame({
    "numero_controle_pncp": resultados["numero_controle_pncp"],
    "homologacao": homologacao,
}).merge(
    pd.DataFrame({"numero_controle_pncp": compras["numero_controle_pncp"], "publicacao": publicacao}),
    on="numero_controle_pncp",
    how="left",
)

antes = datas["homologacao"] < datas["publicacao"]
mesmo_dia = datas["homologacao"] == datas["publicacao"]

print("Resultados:", len(datas))
print("Homologação ANTES da publicação (impossível):", antes.sum())
print("Homologação no MESMO dia da publicação (válida, ciclo 0):", mesmo_dia.sum())


Resultados: 15896
Homologação ANTES da publicação (impossível): 234
Homologação no MESMO dia da publicação (válida, ciclo 0): 1239


**Resultado:**

**Formatos:** 7 formatos em cada coluna. O ISO aparece com hora na publicação (`2025-01-06T11:51:06`) e sem hora no resultado (`2025-03-10`); ao descartar a hora (10 primeiros caracteres), os dois se igualam. Com a lista de 6 formatos, 100% das datas foram convertidas (nenhuma falha).

**Cronologia (regra 7):** dos 15.896 resultados,
- 234 têm homologação **anterior** à publicação: data impossível. Na fato, o item continua (conta no volume), mas fica sem data de homologação e sem ciclo;
- 1.239 foram homologados no **mesmo dia** da publicação: válidos, com ciclo de 0 dias.

Esta contagem é por resultado (todos os classificados). Na fato, com um vencedor por item, o número de itens sem ciclo é diferente.


---
## 4. Nulos disfarçados


In [10]:
NULOS = {"N/A", "NULL", "NONE", "NAN", "-", "0", "SEM INFORMAÇÃO", "SEM INFORMACAO",
         "NÃO INFORMADO", "NAO INFORMADO", "NÃO SE APLICA", "NAO SE APLICA"}

COLUNAS_NUMERICAS = {"quantidade", "valor_unitario_estimado", "valor_total_estimado",
                     "quantidade_homologada", "valor_unitario_homologado", "valor_total_homologado"}

def auditar_nulos(tabela, nome_tabela):
    linhas = []
    for coluna in tabela.columns:
        texto = tabela[coluna].str.strip().str.upper()
        nulos_da_coluna = NULOS - {"0"} if coluna in COLUNAS_NUMERICAS else NULOS
        vazios = tabela[coluna].isna().sum()
        disfarcados = texto.isin(nulos_da_coluna).sum()
        if vazios + disfarcados > 0:
            linhas.append({
                "tabela": nome_tabela,
                "coluna": coluna,
                "vazios reais": vazios,
                "disfarçados": disfarcados,
                "total sem informação": vazios + disfarcados,
                "% da coluna": round((vazios + disfarcados) / len(tabela) * 100, 1),
            })
    return pd.DataFrame(linhas)


In [11]:
pd.concat([
    auditar_nulos(compras, "compras"),
    auditar_nulos(itens, "itens"),
    auditar_nulos(resultados, "resultados"),
], ignore_index=True)


,tabela,coluna,vazios reais,disfarçados,total sem informação,% da coluna
0,compras,processo,28,49,77,3.8
1,compras,modo_disputa,26,941,967,47.4
2,compras,data_abertura_proposta,617,0,617,30.2
3,compras,data_encerramento_proposta,617,0,617,30.2
4,itens,categoria_item,0,7077,7077,100.0
5,itens,unidade_medida,9,0,9,0.1
6,itens,situacao_item,92,128,220,3.1
7,itens,catalogo_codigo_item,7068,0,7068,99.9
8,itens,criterio_julgamento,0,1883,1883,26.6
9,resultados,ordem_classificacao_srp,11976,0,11976,75.3


**Resultado:** 12 colunas têm vazios reais ou nulos disfarçados. A comparação foi feita em maiúsculas e sem espaços, com a mesma lista de nulos do kit; o `0` não foi considerado nulo nas colunas numéricas.

**Nem todo vazio é problema:**
- `ordem_classificacao_srp` (75,3% vazia): compras sem Registro de Preços não têm ranking; pela regra, vazio vale 1.
- `data_cancelamento` (99,6% vazia): vazio significa que o resultado **não** foi cancelado.

**Falta de informação de fato:**
- `categoria_item` (100%, toda "Não se aplica") e `catalogo_codigo_item` (99,9%): colunas sem utilidade para a análise.
- `porte` (30,8%): limita uma eventual análise por porte de empresa.
- `modo_disputa` (47,4%) e `criterio_julgamento` (26,6%): em boa parte "Não se aplica", coerente com compras sem disputa (Dispensa, Inexigibilidade).
- `processo`, `situacao_item`, `unidade_medida` e as datas de proposta: não são usadas nas 6 perguntas.

Nenhuma coluna usada nos cálculos (valores, quantidades, datas de publicação e de resultado, documentos) tem vazios.


---
## 5. Encoding e nomes


In [12]:
padrao_quebrado = r"[ÃÂ][\x80-\xbf]"

def auditar_encoding(tabela, nome_tabela):
    contagem = tabela.apply(lambda coluna: coluna.str.contains(padrao_quebrado, na=False)).sum()
    contagem = contagem[contagem > 0]
    return pd.DataFrame({"tabela": nome_tabela, "coluna": contagem.index, "textos quebrados": contagem.values})

pd.concat([
    auditar_encoding(compras, "compras"),
    auditar_encoding(itens, "itens"),
    auditar_encoding(resultados, "resultados"),
], ignore_index=True)


,tabela,coluna,textos quebrados
0,compras,objeto_compra,43
1,compras,municipio,28
2,compras,nome_unidade,14
3,itens,descricao,116
4,resultados,nome_fornecedor,1


In [13]:
cnpj_limpo = compras["cnpj_orgao"].str.replace(r"\D", "", regex=True).str.zfill(14)

nomes_por_cnpj = compras.groupby(cnpj_limpo)["razao_social_orgao"].nunique()

print("CNPJs de órgão:", len(nomes_por_cnpj))
print("CNPJs com mais de um nome:", (nomes_por_cnpj > 1).sum())


CNPJs de órgão: 370
CNPJs com mais de um nome: 67


In [14]:
cnpj_exemplo = nomes_por_cnpj.sort_values().index[-1]
compras.loc[cnpj_limpo == cnpj_exemplo, "razao_social_orgao"].map(repr).value_counts()


razao_social_orgao
'COMANDO DA AERONAUTICA'           76
'comando da aeronautica'            4
'  COMANDO  DA AERONAUTICA\xa0'     3
'Comando Da Aeronautica'            1
Name: count, dtype: int64

In [15]:
def padronizar_simples(serie):
    return serie.str.strip().str.upper().str.replace(r"\s+", " ", regex=True)

for tabela, coluna in [(compras, "razao_social_orgao"), (resultados, "nome_fornecedor")]:
    antes = tabela[coluna].nunique()
    depois = padronizar_simples(tabela[coluna]).nunique()
    print(f"{coluna}: {antes} nomes distintos -> {depois} após padronizar ({antes - depois} variações)")


razao_social_orgao: 440 nomes distintos -> 352 após padronizar (88 variações)
nome_fornecedor: 10707 nomes distintos -> 10166 após padronizar (541 variações)


**Resultado:**

**Acentos quebrados** (UTF-8 lido como Latin-1) em 5 colunas: `objeto_compra` (43), `municipio` (28), `nome_unidade` (14), `descricao` (116) e `nome_fornecedor` (1).

**Nomes despadronizados:** 67 dos 370 CNPJs de órgão (18%) aparecem com mais de um nome. Exemplo: o Comando da Aeronáutica aparece como `COMANDO DA AERONAUTICA`, `comando da aeronautica`, `Comando Da Aeronautica` e `  COMANDO  DA AERONAUTICA\xa0`. As diferenças são de maiúsculas/minúsculas, espaços nas pontas, espaços duplos e espaço invisível.

Padronizando (maiúsculas, sem espaços extras):
- `razao_social_orgao`: 440 → 352 nomes distintos (88 variações);
- `nome_fornecedor`: 10.707 → 10.166 nomes distintos (541 variações).

Há menos nomes (352) que CNPJs de órgão (370) porque o mesmo órgão tem várias unidades compradoras. Por isso a pergunta 4 agrupa pelo nome padronizado, e a contagem de unidades usa CNPJs distintos (regra 10).
